# Solution E3 · A colleague discovers that wines are a one-dimensional problem

**Type**: Find the bug

## Objective

Find out why a PCA on the wines gives 99.8% of the variance to the first component ("the dataset is almost one-dimensional") and a classifier on that single component reaches 0.70: the PCA was done **on the raw data**, where `proline` (scale of hundreds) decides alone. Fix it by
standardizing and verify how false the conclusion was.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import load_wine
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y = load_wine(return_X_y=True, as_frame=True)
cv = StratifiedKFold(5, shuffle=True, random_state=0)

## Commented solution

In [ ]:
pca = PCA().fit(X)
raw_pc1_ratio = pca.explained_variance_ratio_[0]
raw_1_acc = cross_val_score(make_pipeline(PCA(1), LogisticRegression(max_iter=3000)), X, y, cv=cv).mean()
print(f"variance explained by PC1: {raw_pc1_ratio:.3f} -> 'the problem is one-dimensional'. Accuracy with 1 component: {raw_1_acc:.3f}")

**Where the bug is.** PCA maximizes the variance, and the variance depends on the **units**: `proline` has a standard deviation of 315 against 0.1-15 for the others. The first component is `proline` and nothing else. The 99.8% does not say that the wines are one-dimensional: it says that a column has
an enormous scale.

In [ ]:
pc1_column = X.columns[np.abs(pca.components_[0]).argmax()]
proline_variance_share = X["proline"].var() / X.var().sum()
print(f"column with the largest loading in PC1: {pc1_column} | share of the total variance due to proline: {proline_variance_share:.4f}")

In [ ]:
scaled = PCA().fit(StandardScaler().fit_transform(X))
scaled_pc1_ratio = scaled.explained_variance_ratio_[0]
scaled_k_for_90 = int(np.searchsorted(np.cumsum(scaled.explained_variance_ratio_), 0.9)) + 1
scaled_1_acc = cross_val_score(make_pipeline(StandardScaler(), PCA(1), LogisticRegression(max_iter=3000)), X, y, cv=cv).mean()
scaled_2_acc = cross_val_score(make_pipeline(StandardScaler(), PCA(2), LogisticRegression(max_iter=3000)), X, y, cv=cv).mean()
print(f"after scaling: PC1 explains {scaled_pc1_ratio:.1%}, {scaled_k_for_90} components are needed for 90%")
print(f"accuracy with 1 component: {scaled_1_acc:.3f} | with 2: {scaled_2_acc:.3f} (raw, 1 component: {raw_1_acc:.3f})")

## Check

In [ ]:
with verify():
    check(raw_pc1_ratio > 0.99 and raw_1_acc < 0.75, "the colleague: PC1 with over 99% of the variance and accuracy below 0.75", "don't modify the colleague's code")
    check(pc1_column == "proline" and proline_variance_share is not None and proline_variance_share > 0.99, "PC1 is the column `proline`, which has over 99% of the total variance", "X.var() and components_[0]")
    check(scaled_pc1_ratio is not None and 0.3 < scaled_pc1_ratio < 0.4 and scaled_k_for_90 == 8, "after scaling PC1 explains 36%, and 8 components are needed for 90%", "StandardScaler before PCA")
    check(scaled_2_acc is not None and scaled_2_acc > 0.95 and scaled_2_acc > raw_1_acc + 0.2, "with two components of the standardized data the accuracy is over 0.95 (against 0.70)", "make_pipeline(StandardScaler(), PCA(2), LogisticRegression())")
    check(scaled_1_acc > raw_1_acc + 0.1, "even with a single component scaling improves by over 0.1", "compare scaled_1_acc and raw_1_acc")
done()

## What you learned

- **On unstandardized data PCA measures the units, not the data.** The PC1 of the raw wines is `proline` (99.8% of the total variance due to a single column). A "99.8%" is a warning sign, not a result: when a component explains almost everything, look at its loadings right away.
- **After scaling the true structure emerges:** PC1 explains 36%, 8 components are needed for 90%, and with **two** components the accuracy is 0.96 (against 0.70 with the "99.8% component"). The colleague's conclusion ("the wines are one-dimensional") was false, and an analysis built on top of it would have gone in the wrong direction.
- The rule: **standardize before a PCA** when the columns have different units. The exception is when the columns are already homogeneous (pixels, measurements with the same unit) and you want to preserve the differences in variance (see B1).
- This bug is a twin of E3 of module 5 (the `Normalizer` instead of the `StandardScaler`): in both cases there is no error but a number that "looks" like information. The check is always the same: **look at the loadings and compare with a correct baseline.**

## Common pitfall

Reporting the explained variance of a PCA without saying whether the data were standardized. It is not interpretable: it holds for those units, not for the phenomenon.

## Interview question

<details><summary>Why do you standardize before a PCA?</summary>

Because PCA looks for the directions of maximum variance, and the variance depends on the units of measure: a column expressed in small units (hence with large numbers) dominates the components. By standardizing every column has variance 1 and PCA works on the correlation matrix, that is on the relationships
between the variables. If the columns already have the same unit and the differences in variance are informative (pixels), I don't standardize.

</details>